In [28]:
import cv2
import numpy as np
import os

In [29]:
# ==========================================
# PATH SETUP (YOUR PROJECT FOLDER)
# ==========================================
base_path = r"F:\github\BSc_B5_EE4216-main\Mini Project"

input_folder = os.path.join(base_path, "input_images")
output_folder = os.path.join(base_path, "output_images")

os.makedirs(output_folder, exist_ok=True)

In [30]:
# ==========================================
# GET ALL IMAGES
# ==========================================
image_files = [f for f in os.listdir(input_folder)
               if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

if len(image_files) == 0:
    print("❌ No images found in input_images folder")
    exit()

print("✅ Images found:", image_files)

✅ Images found: ['circle1.jpeg', 'circle2.jpeg', 'circle3.jpeg', 'circle4.jpeg', 'circle5.jpeg', 'square1.jpeg', 'square2.jpeg', 'square3.jpeg', 'square4.jpeg', 'square5.jpeg']


In [31]:
# ==========================================
# PROCESS EACH IMAGE
# ==========================================
for file_name in image_files:

    img_path = os.path.join(input_folder, file_name)
    image = cv2.imread(img_path)

    if image is None:
        print("Skipping:", file_name)
        continue

    output = image.copy()

    # ==========================================
    # HSV CONVERSION
    # ==========================================
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

    lower = np.array([5, 40, 40])
    upper = np.array([45, 255, 255])

    mask = cv2.inRange(hsv, lower, upper)

    # ==========================================
    # NOISE REMOVAL
    # ==========================================
    kernel = np.ones((5, 5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=2)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)

    # ==========================================
    # FIND CONTOURS
    # ==========================================
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    valid_contours = [c for c in contours if cv2.contourArea(c) > 2500]

    if len(valid_contours) == 0:
        print(f"{file_name}: No biscuits detected")
        continue

    max_area = max(cv2.contourArea(c) for c in valid_contours)

    intact_count = 0
    broken_count = 0

    # ==========================================
    # ANALYZE EACH CONTOUR
    # ==========================================
    for cnt in valid_contours:

        area = cv2.contourArea(cnt)
        perimeter = cv2.arcLength(cnt, True)

        if perimeter == 0:
            continue

        circularity = (4 * np.pi * area) / (perimeter * perimeter)

        hull = cv2.convexHull(cnt)
        hull_area = cv2.contourArea(hull)

        if hull_area == 0:
            continue

        solidity = area / hull_area

        x, y, w, h = cv2.boundingRect(cnt)
        aspect_ratio = w / float(h)

        size_ratio = area / max_area

        # ==========================================
        # SCORE SYSTEM
        # ==========================================
        score = 0

        if circularity > 0.78:
            score += 2
        elif circularity > 0.65:
            score += 1

        if solidity > 0.95:
            score += 2
        elif solidity > 0.90:
            score += 1

        if size_ratio > 0.80:
            score += 2
        elif size_ratio > 0.60:
            score += 1

        if 0.85 < aspect_ratio < 1.15:
            score += 1

        # ==========================================
        # FINAL CLASSIFICATION
        # ==========================================
        if score >= 6:
            label = "Intact Biscuit"
            color = (0, 255, 0)
            intact_count += 1
        else:
            label = "Broken Biscuit"
            color = (0, 0, 255)
            broken_count += 1

        # DRAW RESULTS
        cv2.drawContours(output, [cnt], -1, color, 2)
        cv2.rectangle(output, (x, y), (x + w, y + h), color, 2)
        cv2.putText(output, label, (x, y - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

    # ==========================================
    # SAVE OUTPUT
    # ==========================================
    name, ext = os.path.splitext(file_name)
    save_path = os.path.join(output_folder, f"result_{name}.jpg")

    cv2.imwrite(save_path, output)

    print(f"{file_name} -> Intact: {intact_count}, Broken: {broken_count}")

print("✅ Processing completed for all images.")

circle1.jpeg -> Intact: 4, Broken: 6
circle2.jpeg -> Intact: 4, Broken: 8
circle3.jpeg -> Intact: 4, Broken: 8
circle4.jpeg -> Intact: 3, Broken: 7
circle5.jpeg -> Intact: 4, Broken: 9
square1.jpeg -> Intact: 3, Broken: 5
square2.jpeg -> Intact: 3, Broken: 5
square3.jpeg -> Intact: 3, Broken: 5
square4.jpeg -> Intact: 3, Broken: 5
square5.jpeg -> Intact: 3, Broken: 6
✅ Processing completed for all images.
